# Analysis: same-location 2D model â€” delay-period tuning

Loads `stage1_topo_sameloc.pt` (binary go/no-go + spatial readout, NO COM).
Question: does the neuron best tuned to the presented location stay active ABOVE
its pre-cue baseline through the delay (spatial working memory)?

In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
import numpy as np, torch
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy import stats as sp_stats
from src.model_topo import BioLeakyRNNTopo
from src.env import CuedTargetSameLoc2D
from src.analysis import collect_trials_spatial, filter_trials
device = 'cpu'

def make_model():
    return BioLeakyRNNTopo(input_size=7, hidden_size=180, output_size=2, dt=20.0, tau=100.0,
        activation='softplus', sigma_rec=0.10, rec_init='diag', use_ei=True, exc_ratio=0.80,
        use_dale=True, mask_seed=42, sheet_side=12, tau_ee=0.25, tau_ie=0.32, tau_ei=0.64,
        tau_ii=0.64, rf_sigma=0.3, tau_e_range=(50,150), tau_i_range=(10,50), use_adaptation=False).to(device)

model = make_model()
model.load_state_dict(torch.load('checkpoints/stage1_topo_sameloc.pt', weights_only=True)['state_dict'], strict=False)
model.eval(); model.noise_at_eval = True
coords = model.coords.detach().cpu().numpy(); n_exc = int(model.n_exc)

def make_env():
    return CuedTargetSameLoc2D(dt=20, cue_strength=1.0, p_distractor_trial=0.0, distractor_strength=0.0)

torch.manual_seed(0); np.random.seed(0)
trials = collect_trials_spatial(model, make_env, n_trials=800, device=device, head='go_nogo')
corr = filter_trials(trials, outcome='correct')
print('correct:', len(corr), '/', len(trials))

## Location-tuned neuron (gain-removed) + sustained-during-delay test

Best-tuned = neuron most specifically modulated by location (its mean-over-locations
gain subtracted), not the most active neuron. eta^2 reports how selective each E neuron is.

In [ ]:
N_BINS = 8
def theta_bin(pos, n=N_BINS):
    x, y = pos; th = np.arctan2(y, x); return int((th + np.pi) / (2*np.pi) * n) % n

by_bin = defaultdict(list)
for tr in corr:
    by_bin[theta_bin(tr['target_pos'])].append(tr)
bins = [b for b in sorted(by_bin) if len(by_bin[b]) >= 10]

# per-bin per-neuron mean post-target activity, then REMOVE each neuron's mean over
# locations (its gain) -> C is pure location modulation (selectivity, not gain).
A = np.stack([np.stack([tr['h'][tr['target_onset']:tr['target_onset']+10, :n_exc].mean(0)
                        for tr in by_bin[b]]).mean(0) for b in bins])   # [B, n_exc]
C = A - A.mean(0)[None, :]

# location selectivity index eta^2 per E-neuron (between-location var / total var)
eta = []
for i in range(n_exc):
    groups = [np.array([tr['h'][tr['target_onset']:tr['target_onset']+10, i].mean() for tr in by_bin[b]]) for b in bins]
    allv = np.concatenate(groups); gm = allv.mean()
    ssb = sum(len(g)*(g.mean()-gm)**2 for g in groups); sst = ((allv-gm)**2).sum()
    eta.append(ssb/sst if sst > 0 else 0.0)
eta = np.array(eta)
print('eta^2 location selectivity: median=%.3f  p90=%.3f  #(eta>0.5)=%d / %d E neurons'
      % (np.median(eta), np.percentile(eta, 90), (eta > 0.5).sum(), n_exc))

# best-tuned = MOST LOCATION-MODULATED neuron at the rep bin (gain removed), not argmax activity
rep = max(bins, key=lambda b: len(by_bin[b]))
bt = by_bin[rep]
best = int(np.argmax(C[bins.index(rep)]))
print('rep angular bin', rep, '| n', len(bt), '| best location-tuned neuron', best,
      '| eta^2=%.2f' % eta[best], '| coords', np.round(coords[best], 2))

# sustained-during-delay test for that (selective) neuron
base, delay = [], []
for tr in bt:
    h = tr['h']; c = tr['cue_onset']; t0 = tr['target_onset']
    if c < 3 or t0 - c < 8:
        continue
    base.append(h[:c, best].mean()); delay.append(h[c+5:t0, best].mean())
base = np.array(base); delay = np.array(delay)
tstat, pval = sp_stats.ttest_rel(delay, base)
print('baseline=%.3f  delay=%.3f  delta=%+.3f  paired-t p=%.2g  (n=%d)'
      % (base.mean(), delay.mean(), delay.mean()-base.mean(), pval, len(base)))
print('=>', 'SUSTAINED above baseline' if (delay.mean() > base.mean() and pval < 0.05) else 'NOT sustained')

## Cue->delay activity trace of the best-tuned neuron

In [ ]:
# cue-aligned average, truncated to the shortest cue->target interval in the bin
Lmin = min(tr['target_onset'] - tr['cue_onset'] for tr in bt)
segs = []
for tr in bt:
    c = tr['cue_onset']
    s = tr['h'][c-5:c+Lmin+1, best]
    if len(s) == 5 + Lmin + 1:
        segs.append(s)
seg = np.stack(segs).mean(0)
rel_ms = (np.arange(-5, Lmin+1)) * 20

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rel_ms, seg, lw=2, color='steelblue', label='best-tuned neuron h(t)')
ax.axhline(base.mean(), ls='--', color='grey', label='pre-cue baseline')
ax.axvline(0, ls=':', color='green', label='cue onset')
ax.axvline(Lmin*20, ls=':', color='red', label='earliest target onset')
ax.set_xlabel('time rel. cue onset (ms)'); ax.set_ylabel('activity')
ax.set_title('Best-tuned neuron: cue -> delay (angular bin %d)' % rep)
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('figures/sameloc_best_neuron_delay.png', dpi=140); plt.show()

## Selectivity map + tuning curves

In [ ]:
# eta^2 selectivity across the E sheet (each neuron at its coords, coloured by selectivity)
fig, ax = plt.subplots(figsize=(5.5, 5))
sc = ax.scatter(coords[:n_exc, 0], coords[:n_exc, 1], c=eta, cmap='viridis',
                s=70, vmin=0, vmax=1, edgecolors='k', lw=0.3)
plt.colorbar(sc, ax=ax, label='eta^2 (location selectivity)')
ax.set_title('Location selectivity across the E sheet'); ax.set_aspect('equal')
ax.set_xlabel('sheet x'); ax.set_ylabel('sheet y')
plt.tight_layout(); plt.savefig('figures/sameloc_selectivity_map.png', dpi=140); plt.show()

In [ ]:
# tuning curves of the 6 most location-selective E neurons (mean post-target activity vs angle)
deg = np.array([(b + 0.5) * 360.0 / N_BINS - 180.0 for b in bins])
topk = np.argsort(-eta)[:6]
fig, ax = plt.subplots(figsize=(7, 4))
for i in topk:
    ax.plot(deg, A[:, i], marker='o', label='n%d (eta2=%.2f)' % (i, eta[i]))
ax.set_xlabel('target angle (deg)'); ax.set_ylabel('mean post-target activity')
ax.set_title('Tuning curves: 6 most location-selective E neurons'); ax.legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig('figures/sameloc_tuning_curves.png', dpi=140); plt.show()

## Selectivity comparison: binary ring model vs sameloc 2D model

Recomputes eta^2 for `stage2_topo` (binary ring task) with the **identical 8-bin method** used above, then overlays the two distributions. Caveat: the ring task varies location along 1 DOF (angle, fixed radius) so angular binning captures all of its location variance; the sameloc 2D task varies 2 DOF (angle+radius), so angle-only binning pushes radius variance into the residual and mechanically deflates eta^2. Both models nonetheless carry strong single-neuron selectivity.

In [ ]:
# Recompute eta^2 for the binary model with the IDENTICAL 8-bin method, then compare.
from src.env import CuedTargetWithDistractorsV3

def eta_from_trials(trials, n_exc, coords=None, n_bins=N_BINS):
    """8-bin angular location selectivity (eta^2) + gain/selectivity/topo neuron picks."""
    bb = defaultdict(list)
    for tr in trials:
        bb[theta_bin(tr['target_pos'], n_bins)].append(tr)
    bns = [b for b in sorted(bb) if len(bb[b]) >= 10]
    A = np.stack([np.stack([tr['h'][tr['target_onset']:tr['target_onset']+10, :n_exc].mean(0)
                            for tr in bb[b]]).mean(0) for b in bns])   # [B, n_exc]
    C = A - A.mean(0)[None, :]   # gain removed -> pure location modulation
    eta = np.zeros(n_exc)
    for i in range(n_exc):
        groups = [np.array([tr['h'][tr['target_onset']:tr['target_onset']+10, i].mean() for tr in bb[b]]) for b in bns]
        allv = np.concatenate(groups); gm = allv.mean()
        ssb = sum(len(g)*(g.mean()-gm)**2 for g in groups); sst = ((allv-gm)**2).sum()
        eta[i] = ssb/sst if sst > 0 else 0.0
    sel_argmax = [int(np.argmax(A[k])) for k in range(len(bns))]
    sel_mod    = [int(np.argmax(C[k])) for k in range(len(bns))]
    out = dict(eta=eta, n_bins=len(bns),
               argmax_distinct=len(set(sel_argmax)),
               mod_distinct=len(set(sel_mod)))
    if coords is not None:
        sel_rf = []
        for b in bns:
            tp = np.mean([tr['target_pos'] for tr in bb[b]], axis=0)
            sel_rf.append(int(np.argmin(((coords[:n_exc] - tp)**2).sum(1))))
        out['rf_distinct'] = len(set(sel_rf))
    return out

# sameloc: reuse trials collected in cell 1
res_same = eta_from_trials(corr, n_exc, coords=coords)
eta_same = res_same['eta']

# binary ring model (same architecture, different checkpoint + env)
m_bin = make_model()
m_bin.load_state_dict(torch.load('checkpoints/stage2_topo.pt', weights_only=True)['state_dict'], strict=False)
m_bin.eval(); m_bin.noise_at_eval = True
def make_env_binary():
    return CuedTargetWithDistractorsV3(dt=20, cue_strength=1.0, p_distractor_trial=0.6,
                                       distractor_strength=1.0, continuous_locations=True)
torch.manual_seed(0); np.random.seed(0)
tr_bin = collect_trials_spatial(m_bin, make_env_binary, n_trials=800, device=device, head='go_nogo')
corr_bin = filter_trials(tr_bin, outcome='correct')
res_bin = eta_from_trials(corr_bin, n_exc)
eta_bin = res_bin['eta']
print('binary  : median=%.3f #(>0.5)=%d/%d argmax=%d/%d mod=%d/%d'
      % (np.median(eta_bin), (eta_bin>0.5).sum(), n_exc, res_bin['argmax_distinct'], res_bin['n_bins'], res_bin['mod_distinct'], res_bin['n_bins']))
print('sameloc : median=%.3f #(>0.5)=%d/%d argmax=%d/%d mod=%d/%d rf=%d/%d'
      % (np.median(eta_same), (eta_same>0.5).sum(), n_exc, res_same['argmax_distinct'], res_same['n_bins'], res_same['mod_distinct'], res_same['n_bins'], res_same['rf_distinct'], res_same['n_bins']))

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.2))

# left: overlaid eta^2 distributions over the 144 E neurons
bins_h = np.linspace(0, 1, 21)
axL.hist(eta_bin,  bins=bins_h, alpha=0.55, color='C3', label='binary ring')
axL.hist(eta_same, bins=bins_h, alpha=0.55, color='C0', label='sameloc 2D')
axL.axvline(np.median(eta_bin),  color='C3', ls='--', lw=2)
axL.axvline(np.median(eta_same), color='C0', ls='--', lw=2)
axL.axvline(0.5, color='grey', ls=':', lw=1.5)
axL.set_xlabel('eta^2 (location selectivity, 8 angular bins)')
axL.set_ylabel('# E neurons'); axL.set_title('Single-neuron location selectivity')
axL.legend()

# right: normalised summary bars (everything on a 0-1 scale)
labels = ['median\neta^2', '#(eta>0.5)\n/144', 'argmax\ndistinct/8', 'modulation\ndistinct/8']
bin_vals  = [np.median(eta_bin),  (eta_bin>0.5).mean(),  res_bin['argmax_distinct']/res_bin['n_bins'],  res_bin['mod_distinct']/res_bin['n_bins']]
same_vals = [np.median(eta_same), (eta_same>0.5).mean(), res_same['argmax_distinct']/res_same['n_bins'], res_same['mod_distinct']/res_same['n_bins']]
x = np.arange(len(labels)); w = 0.38
axR.bar(x-w/2, bin_vals,  w, color='C3', label='binary ring')
axR.bar(x+w/2, same_vals, w, color='C0', label='sameloc 2D')
axR.set_xticks(x); axR.set_xticklabels(labels, fontsize=8)
axR.set_ylim(0, 1.08); axR.set_ylabel('fraction (0-1)')
axR.set_title('Selectivity summary (normalised)'); axR.legend(loc='upper right')
for xi, (bv, sv) in enumerate(zip(bin_vals, same_vals)):
    axR.text(xi-w/2, bv+0.02, '%.2f'%bv, ha='center', fontsize=7)
    axR.text(xi+w/2, sv+0.02, '%.2f'%sv, ha='center', fontsize=7)
axR.text(0.02, 0.99, 'sameloc nearest-RF distinct = %d/%d' % (res_same['rf_distinct'], res_same['n_bins']),
         transform=axR.transAxes, fontsize=8, va='top', color='C0')

plt.tight_layout(); plt.savefig('figures/sameloc_vs_binary_selectivity.png', dpi=140); plt.show()

## Best location-tuned neuron: cue -> delay -> target (both models)

For the representative location, pick the neuron that most **specifically** codes it (gain-removed modulation = argmax of `C`). Plot its cue-aligned mean activity through the delay, truncated at the **earliest** target onset so every trial is still in the delay at every plotted time point. Question: after the cue, does activity return to the pre-cue baseline, or stay **sustained above baseline** right up to target? Left = binary ring model, right = sameloc 2D.

In [ ]:
def best_neuron_delay_trace(trials, n_exc, n_bins=N_BINS, pre=5):
    """Cue-aligned mean activity of the most location-specific neuron at the rep bin,
    truncated at the earliest cue->target interval (entirely delay). Returns plot data
    + pre-cue baseline vs delay paired-t test."""
    bb = defaultdict(list)
    for tr in trials:
        bb[theta_bin(tr['target_pos'], n_bins)].append(tr)
    bns = [b for b in sorted(bb) if len(bb[b]) >= 10]
    A = np.stack([np.stack([tr['h'][tr['target_onset']:tr['target_onset']+10, :n_exc].mean(0)
                            for tr in bb[b]]).mean(0) for b in bns])
    C = A - A.mean(0)[None, :]
    rep = max(bns, key=lambda b: len(bb[b]))
    best = int(np.argmax(C[bns.index(rep)]))
    bt = bb[rep]
    Lmin = min(tr['target_onset'] - tr['cue_onset'] for tr in bt)
    segs = []
    for tr in bt:
        c = tr['cue_onset']
        s = tr['h'][c-pre:c+Lmin+1, best]
        if len(s) == pre + Lmin + 1:
            segs.append(s)
    segs = np.stack(segs)
    seg = segs.mean(0); sem = segs.std(0) / np.sqrt(len(segs))
    rel_ms = np.arange(-pre, Lmin+1) * 20
    base, delay = [], []
    for tr in bt:
        h = tr['h']; c = tr['cue_onset']; t0 = tr['target_onset']
        if c < 3 or t0 - c < 8:
            continue
        base.append(h[:c, best].mean()); delay.append(h[c+5:t0, best].mean())
    base = np.array(base); delay = np.array(delay)
    tstat, pval = sp_stats.ttest_rel(delay, base)
    return dict(best=best, rep=rep, seg=seg, sem=sem, rel_ms=rel_ms, Lmin=Lmin,
                baseline=base.mean(), delay_mean=delay.mean(), pval=pval, n=len(segs))

trc_bin  = best_neuron_delay_trace(corr_bin, n_exc)
trc_same = best_neuron_delay_trace(corr, n_exc)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3), sharey=False)
for ax, trc, title, col in [(axes[0], trc_bin, 'binary ring model', 'C3'),
                            (axes[1], trc_same, 'sameloc 2D model', 'C0')]:
    rel, seg, sem = trc['rel_ms'], trc['seg'], trc['sem']
    ax.plot(rel, seg, lw=2, color=col, label='best-tuned neuron h(t)')
    ax.fill_between(rel, seg-sem, seg+sem, color=col, alpha=0.2)
    ax.axhline(trc['baseline'], ls='--', color='grey', label='pre-cue baseline')
    ax.axvline(0, ls=':', color='green', label='cue onset')
    ax.axvline(trc['Lmin']*20, ls=':', color='red', label='earliest target')
    ax.axvspan(0, trc['Lmin']*20, color='gold', alpha=0.07)
    verdict = 'SUSTAINED' if (trc['delay_mean'] > trc['baseline'] and trc['pval'] < 0.05) else 'returns to baseline'
    ax.set_title('%s\nneuron %d | base=%.2f delay=%.2f | %s (p=%.1g)'
                 % (title, trc['best'], trc['baseline'], trc['delay_mean'], verdict, trc['pval']), fontsize=9)
    ax.set_xlabel('time from cue (ms)'); ax.set_ylabel('activity'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig('figures/best_neuron_delay_both.png', dpi=140); plt.show()
print('binary : base=%.3f delay=%.3f p=%.2g | sameloc: base=%.3f delay=%.3f p=%.2g'
      % (trc_bin['baseline'], trc_bin['delay_mean'], trc_bin['pval'],
         trc_same['baseline'], trc_same['delay_mean'], trc_same['pval']))